In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    classification_report
)
from catboost import CatBoostClassifier, Pool

# 1. Đọc dữ liệu
print("Đang đọc dữ liệu train.csv...")
train = pd.read_csv("train.csv")

target_col = "is_fraud"
id_col = "id"
feature_cols = [col for col in train.columns if col not in [target_col, id_col]]

X = train[feature_cols].copy()
y = train[target_col].astype(int)

# 2. Xác định các cột phân loại (categorical features)
# CatBoost xử lý trực tiếp chuỗi text/categorical rất tốt khi điền missing và chuyển thành str
cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)

# 3. Chia tập Train - Validation (80% train, 20% val với stratify theo nhãn y)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Kích thước X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"Tỉ lệ fraud train: {y_train.mean():.4f}, val: {y_val.mean():.4f}")

Đang đọc dữ liệu train.csv...
Kích thước X_train: (559968, 24), X_val: (139992, 24)
Tỉ lệ fraud train: 0.0167, val: 0.0167


In [2]:
# 4. Khởi tạo và cấu hình mô hình CatBoost
# Dùng auto_class_weights='Balanced' hoặc eval_metric='PRAUC' để tối ưu cho bài toán Fraud imbalanced
model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=6,
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=42,
    early_stopping_rounds=100,
    verbose=100
)

# 5. Huấn luyện mô hình
train_pool = Pool(X_train, y_train, cat_features=cat_cols)
val_pool = Pool(X_val, y_val, cat_features=cat_cols)

print("Bắt đầu huấn luyện CatBoost...")
model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# 6. Đánh giá kết quả trên tập Validation với 4 metrics
val_prob = model.predict_proba(val_pool)[:, 1]
threshold = 0.5
val_pred = (val_prob >= threshold).astype(int)

roc_auc = roc_auc_score(y_val, val_prob)
auc_pr = average_precision_score(y_val, val_prob)
precision = precision_score(y_val, val_pred, zero_division=0)
recall = recall_score(y_val, val_pred, zero_division=0)

print("\n" + "="*40)
print("--- ĐÁNH GIÁ TRÊN TẬP VALIDATION (0.2) ---")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"AUC-PR    : {auc_pr:.4f}")
print(f"Precision : {precision:.4f} (với threshold = {threshold})")
print(f"Recall    : {recall:.4f} (với threshold = {threshold})")
print("="*40)

print("\nBáo cáo chi tiết Classification Report:")
print(classification_report(y_val, val_pred, digits=4))

Bắt đầu huấn luyện CatBoost...
0:	learn: 0.8981131	test: 0.8947303	best: 0.8947303 (0)	total: 534ms	remaining: 13m 20s
100:	learn: 0.9475575	test: 0.9438063	best: 0.9438072 (99)	total: 1m	remaining: 13m 54s
200:	learn: 0.9502020	test: 0.9455521	best: 0.9455521 (200)	total: 2m 14s	remaining: 14m 26s
300:	learn: 0.9526011	test: 0.9461040	best: 0.9461260 (299)	total: 3m 3s	remaining: 12m 9s
400:	learn: 0.9556755	test: 0.9463508	best: 0.9463669 (399)	total: 4m 24s	remaining: 12m 3s
500:	learn: 0.9583287	test: 0.9464684	best: 0.9464684 (500)	total: 5m 3s	remaining: 10m 5s
600:	learn: 0.9606812	test: 0.9465736	best: 0.9465932 (584)	total: 5m 38s	remaining: 8m 26s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9465932421
bestIteration = 584

Shrink model to first 585 iterations.

--- ĐÁNH GIÁ TRÊN TẬP VALIDATION (0.2) ---
ROC-AUC   : 0.9435
AUC-PR    : 0.5099
Precision : 0.1196 (với threshold = 0.5)
Recall    : 0.8443 (với threshold = 0.5)

Báo cáo chi tiết Classificatio

In [3]:
# 7. (Tùy chọn) Dự đoán trên test.csv và xuất submission
try:
    test = pd.read_csv("test.csv")
    X_test = test[feature_cols].copy()
    for col in cat_cols:
        X_test[col] = X_test[col].fillna("Missing").astype(str)
    
    test_pool = Pool(X_test, cat_features=cat_cols)
    test_prob = model.predict_proba(test_pool)[:, 1]
    
    submission = pd.DataFrame({
        id_col: test[id_col],
        target_col: test_prob
    })
    submission.to_csv("submission_catboost.csv", index=False)
    print("Đã lưu kết quả dự đoán vào submission_catboost.csv thành công!")
except Exception as e:
    print("Bỏ qua bước dự đoán test:", e)

Đã lưu kết quả dự đoán vào submission_catboost.csv thành công!
